In [1]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated successfully')

Authenticated successfully


In [2]:
from google.cloud import bigquery
import pandas as pd

project_id = 'faang-portfolio-olist'
client = bigquery.Client(project=project_id)

query = """
SELECT
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp,
    p.payment_value
FROM `faang-portfolio-olist.ecommerce_data.customers` AS c
JOIN `faang-portfolio-olist.ecommerce_data.orders` AS o ON c.customer_id = o.customer_id
JOIN `faang-portfolio-olist.ecommerce_data.payments` AS p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
"""

df = client.query(query).to_dataframe()
df.head()

,customer_unique_id,order_id,order_purchase_timestamp,payment_value
0,f7d7fc0a59ef4363fdce6e3aa069d498,6808f7954e27254ceae6640f4e903775,2017-06-19 12:31:58+00:00,75.28
1,5dbba6c01268a8ad43f79157bf4454a0,5f082c0821ef7e1812362878c7268aef,2017-11-18 00:58:41+00:00,343.08
2,36b1c0516f123351ffa87430416dcae5,b161deecd4d00e692166e541a2bff028,2017-08-22 22:03:32+00:00,321.92
3,721d1092e1a6460c67e6a0e691d899a3,cab041d8d606d6c9d3165da578795340,2017-05-13 13:16:20+00:00,50.14
4,7a2dc4682890550ebe3b8befcea3d55c,feec317d7127a4dc69cfb507f08c8759,2018-03-04 23:38:00+00:00,105.75


In [3]:

df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

snapshot_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days, # Recency
    'order_id': 'count', # Frequency
    'payment_value': 'sum' # Monetary
})

rfm.rename(columns={
    'order_purchase_timestamp': 'Recency',
    'order_id': 'Frequency',
    'payment_value': 'Monetary'
}, inplace=True)

rfm.head()

,Recency,Frequency,Monetary
customer_unique_id,,,
0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90
0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19
0000f46a3911fa3c0805444483337064,537,1,86.22
0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62
0004aac84e0df4da2b147fca70cf8255,288,1,196.89


In [4]:

rfm.to_csv('olist_rfm_segments.csv')
print("File saved successfully!")

File saved successfully!
